# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

### 1. Target distribution

In [3]:
print(fraud["fraud"].value_counts())
print(fraud["fraud"].value_counts(normalize=True))

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64
fraud
0.0    0.912597
1.0    0.087403
Name: proportion, dtype: float64


The dataset is imbalanced because legitimate transactions make up most of the target, while fraudulent transactions are a small minority.

In [4]:
X = fraud.drop(columns="fraud")
y = fraud["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 2–3. Logistic Regression and evaluation

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, recall_score

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_scaled, y_train)
baseline_predictions = baseline_model.predict(X_test_scaled)
baseline_recall = recall_score(y_test, baseline_predictions)

print(classification_report(y_test, baseline_predictions))
print(f"Fraud recall: {baseline_recall:.3f}")

              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98    182519
         1.0       0.90      0.61      0.72     17481

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000

Fraud recall: 0.606


Recall for class 1 is important here because it shows how many fraudulent transactions the model detects.

### 4. Random oversampling

In [6]:
from imblearn.over_sampling import RandomOverSampler

oversampler = RandomOverSampler(random_state=42)
X_train_over, y_train_over = oversampler.fit_resample(X_train_scaled, y_train)
print(y_train_over.value_counts())

oversample_model = LogisticRegression(max_iter=1000)
oversample_model.fit(X_train_over, y_train_over)
oversample_predictions = oversample_model.predict(X_test_scaled)
oversample_recall = recall_score(y_test, oversample_predictions)

print(classification_report(y_test, oversample_predictions))
print(f"Fraud recall: {oversample_recall:.3f}")

fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000

Fraud recall: 0.948


### 5. Random undersampling

In [7]:
from imblearn.under_sampling import RandomUnderSampler

undersampler = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = undersampler.fit_resample(X_train_scaled, y_train)
print(y_train_under.value_counts())

undersample_model = LogisticRegression(max_iter=1000)
undersample_model.fit(X_train_under, y_train_under)
undersample_predictions = undersample_model.predict(X_test_scaled)
undersample_recall = recall_score(y_test, undersample_predictions)

print(classification_report(y_test, undersample_predictions))
print(f"Fraud recall: {undersample_recall:.3f}")

fraud
0.0    69922
1.0    69922
Name: count, dtype: int64


              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000

Fraud recall: 0.947


### 6. SMOTE

In [8]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(y_train_smote.value_counts())

smote_model = LogisticRegression(max_iter=1000)
smote_model.fit(X_train_smote, y_train_smote)
smote_predictions = smote_model.predict(X_test_scaled)
smote_recall = recall_score(y_test, smote_predictions)

print(classification_report(y_test, smote_predictions))
print(f"Fraud recall: {smote_recall:.3f}")

fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000

Fraud recall: 0.948


### Comparison

In [9]:
recall_results = pd.Series({
    "Baseline": baseline_recall,
    "Oversampling": oversample_recall,
    "Undersampling": undersample_recall,
    "SMOTE": smote_recall
}, name="Fraud recall")
recall_results

Baseline         0.605572
Oversampling     0.947886
Undersampling    0.947486
SMOTE            0.948115
Name: Fraud recall, dtype: float64

The balanced models improve fraud recall compared with the baseline, so they catch a larger share of fraudulent transactions. The classification reports should also be checked because higher recall can reduce precision.